# AutoML-M02: LazyPredict

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Screening rápido de ~30 modelos sin tuning usando LazyPredict.
Se ejecuta para **Caso D** y **Caso D_strict**.

## 📊 Qué hace LazyPredict

Entrena automáticamente decenas de clasificadores de scikit-learn con hiperparámetros
por defecto. No es un AutoML completo — es un *screener* que identifica qué familias
de modelos funcionan mejor antes de invertir tiempo en tuning.

## 📝 Nota
Datos auditados (F3-M08). Sin leakage, sin constantes, sin traidoras.

## ⚠️ Requisitos
- **Kernel**: `Python (LazyPredict)` — entorno conda `env_lazypredict`
- Creado con `fautoml_setup_entornos.ipynb` (celda 3)

## 📦 Genera
- `data/automl/results_lazypredict.parquet` — métricas de todos los modelos
- `docs/html/fase_automl/m02_lazypredict.html` — documentación visual

## 🔄 Flujo
M01 (Baselines) → **M02 (LazyPredict)** → M03 (PyCaret)

## 📌 Siguiente
`fautoml_m03_pycaret.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Qué hace: Verifica/instala dependencias faltantes en el entorno conda,
#   carga librerías, detecta rutas y verifica datasets limpios.
# Requisitos: Kernel env_lazypredict (creado con fautoml_setup_entornos.ipynb).
# Nota: Los entornos conda aislados no tienen todas las librerías que
#   src/ necesita (ej: seaborn en graficos.py). La auto-verificación
#   comprueba e instala automáticamente las que falten.
# ============================================================================

import sys, warnings, time, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

# --- Auto-verificación de dependencias ---
# Diccionario: {nombre_import: nombre_pip}
# (nombre_pip puede diferir del import, ej: sklearn → scikit-learn)
DEPENDENCIAS_REQUERIDAS = {
    'lazypredict': 'lazypredict',
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'pyarrow': 'pyarrow',
    'jinja2': 'jinja2',
}

faltantes = []
for modulo, paquete_pip in DEPENDENCIAS_REQUERIDAS.items():
    try:
        __import__(modulo)
    except ImportError:
        faltantes.append(paquete_pip)

if faltantes:
    print(f'⚠️ Instalando dependencias faltantes: {faltantes}')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + faltantes
    )
    print(f'✅ Instaladas: {faltantes}')
else:
    print('✅ Todas las dependencias disponibles')

# --- Detección de entorno (Colab / Local) ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

# --- Imports principales ---
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss)

# --- Verificar LazyPredict ---
from lazypredict.Supervised import LazyClassifier
print('✅ LazyPredict importado correctamente')

# --- Imports del proyecto ---
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

# --- Rutas y constantes ---
RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es
RANDOM_STATE = 42
FRAMEWORK = 'lazypredict'

# DATASETS: Caso D y D_strict (limpios, sin leakage)
DATASETS = {
    'D': RUTA_AUTOML / 'df_exp_automl_D.parquet',
    'D_strict': RUTA_AUTOML / 'dataset_final_tfm.parquet',
}

info_entorno()

# --- Verificación anti-leakage ---
for caso, ruta in DATASETS.items():
    df_tmp = pd.read_parquet(ruta)
    n_cols = df_tmp.shape[1]
    del df_tmp
    print(f'  ✅ Caso {caso}: {ruta.name} ({n_cols} cols) — LIMPIO')
print('✅ Verificación anti-leakage superada')

✅ Todas las dependencias disponibles
✅ LazyPredict importado correctamente
✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Univ

In [2]:
# ============================================================================
# CELDA 2: EJECUTAR LAZYPREDICT (CASO D + D_strict)
# ============================================================================
# Qué hace: Para cada caso, ejecuta LazyClassifier sobre 70/30 split.
#   LazyPredict entrena ~30 clasificadores con parámetros por defecto.
#   Luego recalculamos métricas adicionales (MCC, Kappa, LogLoss) para
#   mantener compatibilidad con el esquema de results_baselines.parquet.
# Genera: DataFrame unificado con todas las métricas por caso.
# ============================================================================

TARGET = 'abandono'
all_results = []

for caso, ruta in DATASETS.items():
    print(f'\n{"="*70}')
    print(f'CASO {caso}: {ruta.name}')
    print(f'{"="*70}')

    # --- Cargar datos ---
    df = pd.read_parquet(ruta)
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    n_rows = len(df) 
    n_cols = len(df.columns) 
    n_feat = len(X.columns) 
    print(f"Dataset: {n_rows:,} filas × {n_cols} columnas")
    print(f"Features: {n_feat}")
    print(f"Target abandono=1: {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")
    
    # --- Split 70/30 estratificado (mismo que M01) ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
    )

    # --- Imputar NaN antes de LazyPredict ---
    # LazyPredict no maneja NaN internamente, así que imputamos con mediana
    # (misma estrategia que M01 para comparabilidad)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )

    # --- Ejecutar LazyClassifier ---
    print(f'\n⚡ Ejecutando LazyClassifier...')
    t0 = time.time()
    clf = LazyClassifier(
        verbose=0,
        ignore_warnings=True,
        custom_metric=None,
        predictions=True     # Devuelve predicciones para cada modelo
    )
    models_df, predictions_df = clf.fit(X_train_imp, X_test_imp, y_train, y_test)
    t_lazy = time.time() - t0
    print(f'  ✅ {len(models_df)} modelos entrenados en {t_lazy:.1f}s')

    # --- Mostrar top 10 ---
    print(f'\n--- TOP 10 CASO {caso} (por F1 Score) ---')
    top10 = models_df.head(10)
    print(top10.to_string())

    # --- Recalcular métricas completas para compatibilidad con M01 ---
    # LazyPredict solo da Accuracy, Balanced Accuracy, ROC AUC, F1, Time.
    # Necesitamos: precision, recall, mcc, kappa, log_loss.
    print(f'\n🔄 Recalculando métricas extendidas...')

    for model_name in models_df.index:
        # Obtener predicciones del modelo
        if model_name in predictions_df.columns:
            y_pred = predictions_df[model_name].values
        else:
            # Si no hay predicciones, usar las métricas de LazyPredict directamente
            row = models_df.loc[model_name]
            all_results.append({
                'caso': caso,
                'framework': FRAMEWORK,
                'model_name': model_name,
                'accuracy': row.get('Accuracy', 0),
                'balanced_accuracy': row.get('Balanced Accuracy', 0),
                'precision': 0,
                'recall': 0,
                'f1': row.get('F1 Score', 0),
                'auc_roc': row.get('ROC AUC', 0),
                'mcc': 0,
                'kappa': 0,
                'log_loss': 999,
                'train_time_sec': round(row.get('Time Taken', 0), 2),
            })
            continue

        row = models_df.loc[model_name]

        # Calcular métricas extendidas
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        mcc = matthews_corrcoef(y_test, y_pred)
        kappa = cohen_kappa_score(y_test, y_pred)

        all_results.append({
            'caso': caso,
            'framework': FRAMEWORK,
            'model_name': model_name,
            'accuracy': row.get('Accuracy', accuracy_score(y_test, y_pred)),
            'balanced_accuracy': row.get('Balanced Accuracy', balanced_accuracy_score(y_test, y_pred)),
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'auc_roc': row.get('ROC AUC', 0),
            'mcc': mcc,
            'kappa': kappa,
            'log_loss': 999,  # LazyPredict no da probabilidades, no podemos calcular log_loss
            'train_time_sec': round(row.get('Time Taken', 0), 2),
        })

    print(f'  ✅ {len(models_df)} modelos procesados para caso {caso}')

# --- DataFrame unificado ---
df_resultados = pd.DataFrame(all_results)
df_resultados = df_resultados.sort_values(['caso', 'f1'], ascending=[True, False]).reset_index(drop=True)

# --- Ranking final por caso ---
for caso in DATASETS.keys():
    print(f'\n--- RANKING CASO {caso} (por F1, top 10) ---')
    mask = df_resultados['caso'] == caso
    print(df_resultados.loc[mask, ['model_name', 'accuracy', 'f1', 'auc_roc', 'mcc', 'train_time_sec']].head(10).to_string(index=False))

print(f'\n📊 Total: {len(df_resultados)} resultados ({len(df_resultados)//2} modelos × 2 casos)')


CASO D: df_exp_automl_D.parquet
Dataset: 33,621 filas × 22 columnas (21 features)
Dataset: 33,621 × 22 cols (21 features)
Target: 9,833 abandono (29.2%)

⚡ Ejecutando LazyClassifier...


  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 6883, number of negative: 16651
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008008 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1533
[LightGBM] [Info] Number of data points in the train set: 23534, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.292470 -> initscore=-0.883416
[LightGBM] [Info] Start training from score -0.883416
  ✅ 27 modelos entrenados en 179.1s

--- TOP 10 CASO D (por F1 Score) ---
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  Time Taken
Model                                                                                    
LGBMClassifier                     0.89               0.85     0.85      0.89        0.69
XGBClassifier                      0.89               0.85     0.85      0.89        1.08
RandomFor

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 6883, number of negative: 16651
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002294 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1096
[LightGBM] [Info] Number of data points in the train set: 23534, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.292470 -> initscore=-0.883416
[LightGBM] [Info] Start training from score -0.883416
  ✅ 27 modelos entrenados en 183.5s

--- TOP 10 CASO D_strict (por F1 Score) ---
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  Time Taken
Model                                                                                    
LGBMClassifier                     0.88               0.84     0.84      0.88        0.79
XGBClassifier                      0.88               0.84     0.84      0.88        0.80
Ra

In [3]:
# ============================================================================
# CELDA 3: GUARDAR RESULTADOS
# ============================================================================
# Qué hace: Guarda métricas en parquet para la comparativa final (M06).
# Genera: data/automl/results_lazypredict.parquet
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_lazypredict.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} filas (caso D + D_strict)')

💾 results_lazypredict.parquet: 54 filas (caso D + D_strict)


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS Y HTML
# ============================================================================
# Qué hace: Genera gráficos de barras comparativos y documentación HTML.
# Genera: docs/html/fase_automl/m02_lazypredict.html
# ============================================================================

graficos_b64 = {}
nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase_automl', modulo_activo='m02'
)

# --- Gráfico 1: Top 15 modelos por F1 (barras horizontales) ---
# Más legible que barras verticales cuando hay muchos modelos
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_plot = df_resultados[mask].head(15).copy()
    df_plot = df_plot.sort_values('f1', ascending=True)  # Para barras horizontales

    fig, ax = plt.subplots(figsize=(12, 8))
    y_pos = np.arange(len(df_plot))

    # Barras de F1 como métrica principal
    bars = ax.barh(y_pos, df_plot['f1'], color='#3182ce', alpha=0.85, height=0.6)

    # Marcar AUC con puntos
    ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=50, zorder=5, label='AUC-ROC')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['model_name'], fontsize=9)
    ax.set_xlabel('Score')
    ax.set_title(f'LazyPredict Caso {caso}: Top 15 Modelos', fontweight='bold', fontsize=14)
    ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1.05)

    # Añadir valores F1 al final de cada barra
    for bar, val in zip(bars, df_plot['f1']):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8, color='#2d3748')

    plt.tight_layout()
    graficos_b64[f'top15_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Gráfico 2: Comparativa métricas (top 5, barras agrupadas) ---
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_plot = df_resultados[mask].head(5).copy()

    metricas_plot = ['accuracy', 'f1', 'auc_roc', 'mcc', 'balanced_accuracy']
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df_plot))
    width = 0.15
    colores = ['#3182ce', '#38a169', '#e53e3e', '#805ad5', '#ed8936']

    for j, (met, col) in enumerate(zip(metricas_plot, colores)):
        ax.bar(x + j*width, df_plot[met], width, label=met, color=col)

    ax.set_xticks(x + width*2)
    ax.set_xticklabels(df_plot['model_name'], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(f'LazyPredict Caso {caso}: Top 5 — Métricas Detalladas', fontweight='bold', fontsize=14)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
    plt.tight_layout()
    graficos_b64[f'barras_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Construir HTML ---
contenido_html = ''

# Sección introductoria
contenido_html += generar_seccion_html('Metodología', f'''
<p><strong>LazyPredict</strong> entrena automáticamente ~30 clasificadores de scikit-learn
con hiperparámetros por defecto. No realiza tuning — su objetivo es identificar
qué <em>familias de modelos</em> son más prometedoras antes de invertir en optimización.</p>
<div style="background:#FFF5F5;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #e53e3e;">
  <strong>⚠️ Nota:</strong> Las métricas son con parámetros por defecto. Los modelos con tuning
  (PyCaret, H2O, AutoGluon) superarán estas cifras.
</div>
<div style="background:#EBF8FF;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #3182ce;">
  <strong>📌 Split:</strong> 70/30 estratificado, random_state=42 (idéntico a M01 para comparabilidad).
  Imputación con mediana.
</div>
''', '🔬')

# Resultados por caso
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_c = df_resultados[mask]
    mejor = df_c.iloc[0]
    n_modelos = len(df_c)

    # Tabla con todos los modelos
    tabla = '<table style="width:100%;border-collapse:collapse;">\n'
    tabla += '<tr style="background:#3182ce;color:white;">'
    for col in ['#', 'Modelo', 'Acc', 'Bal.Acc', 'Prec', 'Recall', 'F1', 'AUC', 'MCC', 'Kappa', 'Tiempo']:
        tabla += f'<th style="padding:8px;text-align:center;font-size:11px;">{col}</th>'
    tabla += '</tr>\n'

    for rank, (i, row) in enumerate(df_c.iterrows(), 1):
        bg = '#f7fafc' if rank % 2 == 0 else 'white'
        # Resaltar top 3
        if rank <= 3:
            medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
            rank_str = medallas[rank]
        else:
            rank_str = str(rank)

        tabla += f'<tr style="background:{bg};">'
        tabla += f'<td style="padding:6px;text-align:center;font-size:11px;">{rank_str}</td>'
        tabla += f'<td style="padding:6px;font-size:11px;">{row["model_name"]}</td>'
        for c in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'mcc', 'kappa']:
            v = row[c]
            color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
            tabla += f'<td style="text-align:center;color:{color};font-size:11px;">{v:.4f}</td>'
        tabla += f'<td style="text-align:center;font-size:11px;">{row["train_time_sec"]:.1f}s</td></tr>\n'
    tabla += '</table>'

    # Gráficos
    graf_top15 = f'<img src="data:image/png;base64,{graficos_b64[f"top15_{caso}"]}" style="max-width:100%;border-radius:8px;">'
    graf_barras = f'<img src="data:image/png;base64,{graficos_b64[f"barras_{caso}"]}" style="max-width:100%;border-radius:8px;">'

    desc_caso = 'Alerta temprana (21 features)' if caso == 'D' else 'Producción (19 features)'
    contenido_html += generar_seccion_html(
        f'⚡ Caso {caso}: {desc_caso}',
        f'<p><strong>🏆 Mejor:</strong> {mejor["model_name"]} '
        f'(F1={mejor["f1"]:.4f}, AUC={mejor["auc_roc"]:.4f}, MCC={mejor["mcc"]:.4f})</p>'
        f'<p><strong>📊 Modelos evaluados:</strong> {n_modelos}</p>\n'
        f'{graf_top15}\n<br>\n{tabla}\n<br>\n{graf_barras}'
    )

# --- Sección comparativa vs baselines ---
# Cargar baselines si existen
ruta_baselines = RUTA_AUTOML / 'results_baselines.parquet'
if ruta_baselines.exists():
    df_base = pd.read_parquet(ruta_baselines)
    comparativa = '<table style="width:100%;border-collapse:collapse;">\n'
    comparativa += '<tr style="background:#3182ce;color:white;">'
    for col in ['Caso', 'Framework', 'Mejor Modelo', 'F1', 'AUC', 'MCC']:
        comparativa += f'<th style="padding:8px;text-align:center;">{col}</th>'
    comparativa += '</tr>\n'

    for caso in DATASETS.keys():
        # Mejor baseline
        mask_b = (df_base['caso'] == caso) & (~df_base['model_name'].str.startswith('Dummy'))
        mejor_b = df_base[mask_b].sort_values('f1', ascending=False).iloc[0]
        comparativa += f'<tr style="background:#f7fafc;">'
        comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
        comparativa += f'<td style="padding:6px;text-align:center;">Baselines (M01)</td>'
        comparativa += f'<td style="padding:6px;">{mejor_b["model_name"]}</td>'
        comparativa += f'<td style="text-align:center;">{mejor_b["f1"]:.4f}</td>'
        comparativa += f'<td style="text-align:center;">{mejor_b["auc_roc"]:.4f}</td>'
        comparativa += f'<td style="text-align:center;">{mejor_b["mcc"]:.4f}</td></tr>\n'

        # Mejor LazyPredict
        mask_lp = df_resultados['caso'] == caso
        mejor_lp = df_resultados[mask_lp].iloc[0]
        comparativa += f'<tr style="background:white;">'
        comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
        comparativa += f'<td style="padding:6px;text-align:center;"><strong>LazyPredict (M02)</strong></td>'
        comparativa += f'<td style="padding:6px;"><strong>{mejor_lp["model_name"]}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_lp["f1"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_lp["auc_roc"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_lp["mcc"]:.4f}</strong></td></tr>\n'

    comparativa += '</table>'
    contenido_html += generar_seccion_html('🔄 Comparativa: Baselines vs LazyPredict', comparativa)
else:
    print('⚠️ results_baselines.parquet no encontrado — se omite comparativa.')

# --- KPIs ---
casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
n_modelos_caso = len(df_resultados[df_resultados['caso']==casos_k[0]])

KPIS = [
    {'valor': str(n_modelos_caso), 'titulo': 'Modelos'},
    {'valor': f'{mejor_d["f1"]:.4f}', 'titulo': f'Mejor F1 (D)'},
    {'valor': f'{mejor_ds["f1"]:.4f}', 'titulo': f'Mejor F1 (D_strict)'},
    {'valor': '2', 'titulo': 'Casos'},
]

# --- Renderizar HTML ---
html = render_base_html(
    titulo='M02: LazyPredict', icono='⚡',
    subtitulo=f'Screening ~{n_modelos_caso} Modelos (Caso D + D_strict)',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + contenido_html,
    notebook_nombre='fautoml_m02_lazypredict.ipynb', notebook_carpeta='fase_automl'
)
ruta_html = RUTA_FASE_AUTOML_HTML / 'm02_lazypredict.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m02_lazypredict.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m02_lazypredict.html


In [5]:
# ============================================================================
# CELDA 5: RESUMEN
# ============================================================================

casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
n_modelos_caso = len(df_resultados[df_resultados['caso']==casos_k[0]])

print('\n' + '='*60)
print('✅ AUTOML-M02 COMPLETADO')
print('='*60)
print(f'Framework: LazyPredict')
print(f'Modelos por caso: {n_modelos_caso}')
print(f'Caso D mejor: {mejor_d["model_name"]} (F1={mejor_d["f1"]:.4f}, AUC={mejor_d["auc_roc"]:.4f})')
print(f'Caso D_strict mejor: {mejor_ds["model_name"]} (F1={mejor_ds["f1"]:.4f}, AUC={mejor_ds["auc_roc"]:.4f})')
print(f'Resultados: {ruta_resultados}')
print(f'HTML: {ruta_html}')
print(f'\n🎯 Siguiente: fautoml_m03_pycaret.ipynb')
print('='*60)


✅ AUTOML-M02 COMPLETADO
Framework: LazyPredict
Modelos por caso: 27
Caso D mejor: LGBMClassifier (F1=0.8063, AUC=0.8545)
Caso D_strict mejor: LGBMClassifier (F1=0.7885, AUC=0.8419)
Resultados: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_lazypredict.parquet
HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m02_lazypredict.html

🎯 Siguiente: fautoml_m03_pycaret.ipynb
